In [1]:
import pandas as pd
import numpy as np
import os
import requests
from datetime import datetime
from pathlib import Path
 
pd.set_option('display.max_columns', None)

In [2]:
# ── 1. PATH CONFIGURATION ───────────────
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parents[2]
 
BASE_DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR    = BASE_DATA_DIR / "raw"
BRONZE_DIR = BASE_DATA_DIR / "bronze"
SILVER_DIR = BASE_DATA_DIR / "silver"
 
RAW_DIR.mkdir(parents=True, exist_ok=True)
BRONZE_DIR.mkdir(parents=True, exist_ok=True)
SILVER_DIR.mkdir(parents=True, exist_ok=True)
 
DATA_URL = "https://www.data.gouv.fr/api/1/datasets/r/06d9816c-1b87-498d-985e-f312acee4f51"
 
path_data_raw     = RAW_DIR / "resultats-par-niveau-subcom-t2-france-entiere.xlsx"
path_rhone_bronze = BRONZE_DIR / "2022-pres-t2-commune-rhone-69-bronze.csv"
path_rhone_silver = SILVER_DIR / "2022-pres-t2-commune-rhone-69-silver.csv"
 
print(f"Raw data path: {path_data_raw}")
print(f"Bronze path:   {path_rhone_bronze}")
print(f"Silver path:   {path_rhone_silver}")


Raw data path: /Users/zainfrayha/Desktop/electio-analytics-poc/data/raw/resultats-par-niveau-subcom-t2-france-entiere.xlsx
Bronze path:   /Users/zainfrayha/Desktop/electio-analytics-poc/data/bronze/2022-pres-t2-commune-rhone-69-bronze.csv
Silver path:   /Users/zainfrayha/Desktop/electio-analytics-poc/data/silver/2022-pres-t2-commune-rhone-69-silver.csv


In [3]:
# =============================================================================
# ── 2. RAW LAYER: Landing Zone 
# =============================================================================
if not os.path.exists(path_data_raw):
    print(f"📥 Downloading source to RAW...")
    resp = requests.get(DATA_URL, timeout=180)
    resp.raise_for_status()
    with open(path_data_raw, "wb") as f:
        f.write(resp.content)
    print(f"✅ Saved to RAW: {path_data_raw}")
else:
    print(f"✅ Raw file already exists in {RAW_DIR}")


✅ Raw file already exists in /Users/zainfrayha/Desktop/electio-analytics-poc/data/raw


In [4]:
# =============================================================================
# ── 3. BRONZE LAYER: Read and fix column names
# =============================================================================
print("\n🛠 Processing RAW -> BRONZE...")
 
df_raw = pd.read_excel(path_data_raw, dtype=str)
 
print(f"Initial shape: {df_raw.shape}")
print(f"Columns with 'Unnamed': {[col for col in df_raw.columns if 'Unnamed' in col]}")
 
# Map Unnamed columns to candidate 2 columns
rename_map = {
    'Unnamed: 26': 'N°Panneau_2',
    'Unnamed: 27': 'Sexe_2',
    'Unnamed: 28': 'Nom_2',
    'Unnamed: 29': 'Prénom_2',
    'Unnamed: 30': 'Voix_2',
    'Unnamed: 31': '% Voix/Ins_2',
    'Unnamed: 32': '% Voix/Exp_2',
}
 
# Only rename columns that exist
rename_map = {k: v for k, v in rename_map.items() if k in df_raw.columns}
df_raw.rename(columns=rename_map, inplace=True)
 
# Also rename candidate 1 columns to be explicit
df_raw.rename(columns={
    'N°Panneau': 'N°Panneau_1',
    'Sexe': 'Sexe_1',
    'Nom': 'Nom_1',
    'Prénom': 'Prénom_1',
    'Voix': 'Voix_1',
    '% Voix/Ins': '% Voix/Ins_1',
    '% Voix/Exp': '% Voix/Exp_1',
}, inplace=True)
 
print(f"Renamed columns: {df_raw.columns.tolist()}")
 
# Filter for Rhône (department 69)
df_raw["Code du département"] = df_raw["Code du département"].astype(str).str.strip().str.zfill(2)
df_bronze = df_raw[df_raw["Code du département"] == "69"].copy()
 
print(f"✅ Filtered to Rhône: {len(df_bronze)} rows")
 
# Add metadata
df_bronze["extraction_source_url"] = DATA_URL
df_bronze["ingestion_timestamp"] = datetime.now().isoformat()
df_bronze["source_file_name"] = os.path.basename(path_data_raw)
 
# Save bronze
df_bronze.to_csv(path_rhone_bronze, index=False, sep=";", encoding="utf-8")
print(f"✅ BRONZE dataset created: {path_rhone_bronze}")



🛠 Processing RAW -> BRONZE...


Initial shape: (35245, 33)
Columns with 'Unnamed': ['Unnamed: 26', 'Unnamed: 27', 'Unnamed: 28', 'Unnamed: 29', 'Unnamed: 30', 'Unnamed: 31', 'Unnamed: 32']
Renamed columns: ['Code du département', 'Libellé du département', 'Code de la commune', 'Libellé de la commune', 'Etat saisie', 'Inscrits', 'Abstentions', '% Abs/Ins', 'Votants', '% Vot/Ins', 'Blancs', '% Blancs/Ins', '% Blancs/Vot', 'Nuls', '% Nuls/Ins', '% Nuls/Vot', 'Exprimés', '% Exp/Ins', '% Exp/Vot', 'N°Panneau_1', 'Sexe_1', 'Nom_1', 'Prénom_1', 'Voix_1', '% Voix/Ins_1', '% Voix/Exp_1', 'N°Panneau_2', 'Sexe_2', 'Nom_2', 'Prénom_2', 'Voix_2', '% Voix/Ins_2', '% Voix/Exp_2']
✅ Filtered to Rhône: 267 rows
✅ BRONZE dataset created: /Users/zainfrayha/Desktop/electio-analytics-poc/data/bronze/2022-pres-t2-commune-rhone-69-bronze.csv


In [5]:
# # =============================================================================
# # ── 4. SILVER LAYER: Data Cleaning & Type Conversion
# # =============================================================================
# print("\n🧹 Processing BRONZE -> SILVER...")
 
# df_silver = df_bronze.copy()
 
# # Identify numeric columns (those with % or vote counts)
# numeric_cols = [col for col in df_silver.columns if any(
#     pattern in col for pattern in ['%', 'Inscrits', 'Abstentions', 'Votants', 
#                                    'Blancs', 'Nuls', 'Exprimés', 'Voix', 
#                                 #    'Code du département', 'Code de la circonscription',
#                                 #    'Code de la commune', 'Code du b.vote'
#                                    ]
# )]
 
# print(f"Converting {len(numeric_cols)} columns to numeric...")
 
# for col in numeric_cols:
#     if col in df_silver.columns:
#         # Replace French decimal separator (comma) with period
#         df_silver[col] = df_silver[col].astype(str).str.replace(',', '.')
#         # Try to convert to float, coerce errors to NaN
#         df_silver[col] = pd.to_numeric(df_silver[col], errors='coerce')
 
# # Check for missing values
# print("\n🚨 Missing values in SILVER:")
# missing = df_silver.isnull().sum()
# missing = missing[missing > 0].sort_values(ascending=False)
# if len(missing) > 0:
#     print(missing)
# else:
#     print("✅ No missing values")
 
# # Save silver
# df_silver.to_csv(path_rhone_silver, index=False, sep=";", encoding="utf-8")
# print(f"\n✅ SILVER dataset created: {path_rhone_silver}")
 
# # Display summary
# print(f"\n📊 SILVER Summary:")
# print(f"Shape: {df_silver.shape}")
# print(f"\nDtypes:")
# print(df_silver.dtypes)
# print(f"\nFirst few rows:")
# print(df_silver.head())
 
